In [4]:
class Variable:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"?{self.name}"

    def __eq__(self, other):
        return isinstance(other, Variable) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

class Constant:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name

    def __eq__(self, other):
        return isinstance(other, Constant) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

class Term:
    def __init__(self, symbol, args=None):
        self.symbol = symbol
        self.args = args if args is not None else []

    def __repr__(self):
        if not self.args:
            return self.symbol
        return f"{self.symbol}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return (isinstance(other, Term) and
                self.symbol == other.symbol and
                self.args == other.args)

    def __hash__(self):
        return hash((self.symbol, tuple(self.args)))

def occurs_check(var, term, subst):
    if var == term:
        return True
    elif isinstance(term, Variable):
        if term in subst:
            return occurs_check(var, subst[term], subst)
    elif isinstance(term, Term):
        return any(occurs_check(var, arg, subst) for arg in term.args)
    return False

def unify(x, y, subst=None):
    if subst is None:
        subst = {}

    if subst is False:
        return False

    if x == y:
        return subst

    if isinstance(x, Variable):
        return unify_var(x, y, subst)

    if isinstance(y, Variable):
        return unify_var(y, x, subst)

    if isinstance(x, Term) and isinstance(y, Term):
        if x.symbol != y.symbol or len(x.args) != len(y.args):
            return False
        for arg_x, arg_y in zip(x.args, y.args):
            subst = unify(arg_x, arg_y, subst)
        return subst

    return False

def unify_var(var, val, subst):
    if var in subst:
        return unify(subst[var], val, subst)
    if isinstance(val, Variable) and val in subst:
        return unify(var, subst[val], subst)
    if occurs_check(var, val, subst):
        return False

    new_subst = subst.copy()
    new_subst[var] = val
    return new_subst

x = Variable('x')
y = Variable('y')
a = Constant('a')
b = Constant('b')

term1 = Term('f', [x, a])
term2 = Term('f', [b, y])

print(f"Unifying: {term1} with {term2}")
result = unify(term1, term2)
print(f"Substitution: {result}\n")

term3 = Term('g', [x])
print(f"Unifying (Occurs check failure case): {x} with {term3}")
result_occurs = unify(x, term3)
print(f"Substitution: {result_occurs}")

Unifying: f(?x, a) with f(b, ?y)
Substitution: {?x: b, ?y: a}

Unifying (Occurs check failure case): ?x with g(?x)
Substitution: False
